# Molecular Hamiltonians on Colab

PySCF HF/DFT &rarr; (active-space) integrals &rarr; qubit Hamiltonian &rarr; exact ADAPT-VQE reference, with every
stage checked against FCI. Run the cells in order; CPU is enough for every example here.

**Three ways to use this project on Colab**

1. *This notebook*, interactively (edit the form fields on the right of each cell).
2. *From your own terminal* with the `colab` CLI, no browser needed, results downloaded and the VM released for you:
   `quantum-workflow-colab examples/lih-bk.json --output results/lih` (see `docs/colab.md`).
3. *Ad hoc* with `colab exec` / `colab run` (see `docs/colab.md` for the exit-code and timeout traps).

Runtimes are temporary: download results or copy them to Drive. TPUs are not supported.

In [ ]:
# @title Get the project { display-mode: "form" }
from pathlib import Path
import subprocess

repo_url = "https://github.com/gutama/colabexercise.git"  # @param {type:"string"}
branch = "main"  # @param {type:"string"}
project = Path("/content/colabexercise")
if not project.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", branch, repo_url,
                    str(project)], check=True)
if not (project / "pyproject.toml").is_file():
    raise RuntimeError("Project files are missing; check repo_url/branch or upload the project.")
print(subprocess.run(["git", "-C", str(project), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
# @title Install (regular install: importable without restarting the runtime)
%pip install -q "/content/colabexercise[qubit]"
import importlib
importlib.invalidate_caches()

## Optional: NVIDIA GPU (GPU4PySCF)
Only for `"device": "gpu"` configs on a GPU runtime with CUDA 12. GPU mode uses density fitting, so its SCF energy
differs slightly from the exact-integral CPU result; exported integrals are always exact. See the
[GPU4PySCF guide](https://pyscf.org/user/gpu.html) for other CUDA versions. Not yet validated on GPU hardware in
this project.

In [ ]:
# @title GPU setup (skipped unless enabled)
INSTALL_GPU = False  # @param {type:"boolean"}
if INSTALL_GPU:
    from quantum_workflow.colab_remote import ensure_gpu4pyscf
    ensure_gpu4pyscf()

## Run a calculation
Pick an example (or edit `data`, a plain dict with the same schema as the JSON files). `run()` refuses to
continue if the exported Hamiltonian does not reproduce the SCF energy on its reference state, or (up to 16
qubits) FCI/CASCI in the right particle-number sector.

In [ ]:
# @title Choose and run { display-mode: "form" }
import json
import os
import time
os.environ.setdefault("OMP_NUM_THREADS", "1")  # reproducible orbitals for degenerate cases
from quantum_workflow.workflow import parse_config, run

example = "lih-bk.json"  # @param ["h2-pyscf.json", "heh-cation-qubit.json", "lih-bk.json", "h2o-cas44.json"]
data = json.loads((project / "examples" / example).read_text())
# data["qubit"] = {"mapping": "jordan_wigner"}   # e.g. switch the mapping
config = parse_config(data, base_dir=project / "examples")
output = Path("/content/results") / f"{Path(example).stem}-{time.strftime('%H%M%S')}"
summary = run(config, output)
print(f"SCF energy        {summary['energy_hartree']:.10f} Ha")
for method, energy in summary["reference_energies_hartree"].items():
    print(f"{method:<17} {energy:.10f}")
if "qubit_hamiltonian" in summary:
    q = summary["qubit_hamiltonian"]
    print(f"\n{q['mapping']}: {q['n_qubits']} qubits, {q['n_terms']} Pauli terms, "
          f"{q['n_qwc_groups']} QWC groups, max weight {q['max_pauli_weight']}")
    print(f"reference bits {q['reference_bits']}  ->  {q['reference_energy_hartree']:.10f} Ha")
    print(f"sector ground state  {q['sector_ground_energy_hartree']}")
print("warnings:", summary["warnings"] or "none")
print("files:", output)

## ADAPT-VQE reference
An exact-statevector ADAPT-VQE run on the exported `qubit_hamiltonian.json` (the file contract, the same way an
external solver would consume it). It is a classical check of the mapping, reference state and operator pool
before you spend simulator or hardware time, not a hardware run.

In [ ]:
# @title ADAPT-VQE { display-mode: "form" }
from quantum_workflow.adapt import run_adapt

max_operators = 10  # @param {type:"integer"}
if (output / "qubit_hamiltonian.json").is_file():
    report = run_adapt(output, max_operators=max_operators)
    for step in report["history"]:
        print(f"{step['operator']:<16} |g|={step['max_gradient']:.2e}  E={step['energy_hartree']:.10f}")
    print(f"error vs exact: {report.get('error_hartree', float('nan')):.2e} Ha, "
          f"chemical accuracy: {report.get('chemical_accuracy')}")
else:
    print("This example has no 'qubit' section.")

## Optional: Quantum ESPRESSO (CPU)
Installs a conda-forge QE 7.6 environment with micromamba (several minutes, ~1 GB) and downloads an example H PBE
PAW pseudopotential. QE treats the molecule in a periodic supercell: converge vacuum and cutoffs yourself, check
the pseudopotential's provenance and functional, and do not compare its pseudopotential energies with all-electron
PySCF energies. QE does not export qubit Hamiltonians here.

In [ ]:
# @title Quantum ESPRESSO (skipped unless enabled)
RUN_QE = False  # @param {type:"boolean"}
if RUN_QE:
    from quantum_workflow.colab_remote import ensure_qe
    from quantum_workflow.workflow import load_config

    os.environ["PATH"] = f"{ensure_qe('/content')}{os.pathsep}{os.environ['PATH']}"
    pseudo = project / "pseudopotentials" / "H.pbe-kjpaw_psl.1.0.0.UPF"
    pseudo.parent.mkdir(exist_ok=True)
    subprocess.run(["curl", "-fsSL", "-o", str(pseudo),
                    "https://pseudopotentials.quantum-espresso.org/upf_files/H.pbe-kjpaw_psl.1.0.0.UPF"],
                   check=True)
    qe_output = Path("/content/results") / f"h2-qe-{time.strftime('%H%M%S')}"
    qe_summary = run(load_config(project / "examples/h2-qe.json"), qe_output)
    print(qe_summary["energy_hartree"], "Ha", qe_summary["warnings"])

## Download
Zips `/content/results` (QE scratch directories excluded). See the README for the file formats and conventions.

In [ ]:
# @title Download results
import shutil
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    staging = Path(tmp) / "results"
    shutil.copytree("/content/results", staging, ignore=shutil.ignore_patterns("scratch"))
    archive = shutil.make_archive("/content/quantum-workflow-results", "zip", staging)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab: archive left at", archive)